# Setting Up
As a reminder from Lesson 1, the first step of using a Jupyter notebook is selecting your environment.

Selecting an environment means you can make use of all of the software in it for your Jupyter notebook, and effectively tells your computer what it has to work with when you ask it for a given function.

To get started, click the environment button in the top right, and select the 2026_figure environment we made in Lesson 0.

Anytime you want to run some code, select the block you want to run, and hit the run button to its left. Hitting `Ctrl + Enter` on your keyboard also runs the block you've got selected. Usually in a Jupyter notebook, blocks depend on the ones above them, so they may not run, or may not run the way you'd expect if the previous ones haven't been run.

---

In [ ]:
# Here once again is our import block.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns

from umap import UMAP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

from warnings import catch_warnings, simplefilter

In [ ]:
# You might notice that our path to our data is different from last week.
# That's because this is what's called a "relative" path.
# A relative path describes where something is in relation to your "working directory".
# In this case, our working directory is generally where the Jupyter notebook we're using is.
DATA_PATH = Path("../lesson_1/iris_dataset.tsv")

# Once again, we're just loading in our data with polars and displaying it to get an overview.
data = pl.read_csv(DATA_PATH, separator="\t")
display(data)

---

# Introduction to UMAP
As I mentioned, UMAP works by compressing our data into a smaller number of components, so that we can then visualize it.
It then sorts these components by their ability to separate out our data, meaning that UMAP 1 does a better job than UMAP 2, and so on.

## Exercise 1
First, we want to consider our data for a moment. Assuming that the thing we care about "predicting" is the type of iris a given set of data describes, how many variables are we working with as input?
- In this case, you can easily count that number from the table above, but the block below shows a way to determine it programmatically.

We next want to determine a more manageable number of dimensions which we can compress our data to. Most of the time, we'll choose either two or three dimensions, and in this case, we'll go with two.

This block then does the following:
- Separate out our outcome we want to predict.
- Run UMAP to compress our predictor variables.
- Plot UMAP 1 and UMAP 2, while coloring each point by its corresponding species.

Once you run the block and observe the results, try to update the plot so that it instead displays UMAP 2 and UMAP 3.
Additionally, think about why we want to make sure we set the aspect ratio of our axes to be equal when plotting a UMAP.

In [ ]:
# This indendented with block is called a "Context Manager" and is used to temporarily apply a condition.
# In this case, we're temporarily suppressing warnings (but not errors) as some are produced due to how we
# are working with our data.
with catch_warnings():
    simplefilter("ignore")
    # This splits our dataset into our list of species and predictors, and encodes our species list as a set of numbers
    species = np.ravel(data.select(pl.col("iris type").rank("dense")))
    predictors = data.select(cs.exclude(pl.col("iris type")))
    # First, we're just printing how many columns are in our predictors
    print(f"Our dataset has {predictors.shape[1]} variables.")

    # The reason we're setting a random state for our umap here is that the way it embeds data is stochastic by default.
    # By setting a fixed seed, we guarantee that for the same data, our umap will always give us the same results.
    umap_embedding = UMAP(n_components=3, random_state=42).fit_transform(X=predictors, y=species)

    # Next, we convert the array we receive from UMAP into a dataframe for ease of use
    # Here we also append back on the species categorization of each iris so that we can plot it
    umap_embedding = pl.concat([data.select(pl.col("iris type")),pl.DataFrame({"umap_1": umap_embedding[:,0], "umap_2": umap_embedding[:,1], "umap_3": umap_embedding[:,2]})], how="horizontal")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal", "box")
    sns.scatterplot(data=umap_embedding, x="umap_1", y="umap_2", hue="iris type")
    plt.xlabel("UMAP 1",fontdict={"fontsize": 16})
    plt.ylabel("UMAP 2",fontdict={"fontsize": 16})
    plt.title("UMAP of Iris Species",fontdict={"fontsize": 18})
    plt.legend(title="Species")
    plt.tight_layout()
    plt.show()

---

# Introduction to Multivariate Linear Regression
Now that we've visualized our iris data, let's see if we can accurately predict what species an iris is, using all of the predictors we have to work with.

## Exercise 2
Once again, we start by separating out our predictors from the thing we want to predict.

The block then does the following:
- It creates a pipeline that first scales and centers our data, and the fits a linear regression to it.
- It fits that linear regression to determine the coefficients for each feature.
- It predicts the species of each iris based on its values.
- Finally it calculates how accurate the model was.

Once you've run the block, try to answer the following questions:
1. Which feature seems to be the most predictive?
2. What happens if we remove the scaling step from our predictive model? Why?

In [ ]:
species = np.ravel(data.select(pl.col("iris type").rank("dense")))
predictors = data.select(cs.exclude(pl.col("iris type")))

pipeline = make_pipeline(StandardScaler(),LinearRegression(n_jobs=-1))

# Here we're training a model to predict species based on our four input variables
regression_classifier = pipeline.fit(X=predictors,y=species)

# BONUS QUESTION: Why are we rounding the predictions from our model here?
predictions = np.round(regression_classifier.predict(X=predictors))
for i, column in enumerate(predictors.columns):
    print(f"The slope for {column} is {pipeline["linearregression"].coef_[i]:.2f}...")

accuracy = np.sum(predictions == species) / species.shape * 100
print(f"\nOur classifier is {accuracy[0]:.2f} % accurate.")

---

# Challenges
Now that we've covered visualizing multivariate data and predicting outcomes using it, try the following exercises.

See if you can do the following in the block below:
1. Plot the UMAP of our data again, but first, scale our predictors using a `StandardScaler()`. What did or didn't change? Why do you think that's the case?

**BONUS CHALLENGES**
1. Think about why it might be a good idea to split your data for training and testing purposes when making predictive models.